# 02 · PV capacity and weather

**Purpose.** Build the two daily driver series the models use: rooftop PV capacity and weather.

**Paper section served.** II Data.

**Inputs** (raw and read-only; paths come from `config.local.yaml`)

- Ausgrid "average electricity use" workbooks by local government area, FY2015 to FY2025: network PV capacity (kWp) as connected at 30 June. FY2024 was published as PDF only, so its total is entered in the configuration.
- BOM Climate Data Online daily files for station 066037 (Sydney Airport AMO): maximum temperature, minimum temperature, global solar exposure.

**Paper artifacts.** None.

**Supporting checks** (`outputs/checks`)

- `check_pv_capacity.csv`: PV capacity read from the workbooks against the expected table.
- `check_weather_missing.csv`: missing days in each weather series, by year.
- `check_weather_quality.csv`: the Bureau's quality flags on the temperature series.

**Data for later notebooks** (`data/interim`)

- `pv_annual.parquet`: PV capacity (kWp) at 30 June, 2015 to 2025.
- `pv_daily.parquet`: daily PV capacity (MW), 2015 to 2025.
- `weather_daily.parquet`: daily weather from 1990, short gaps filled.

**Main process**

1. Set up.
2. Read PV capacity at 30 June and check it against the expected table.
3. Turn it into a daily series.
4. Read the weather as published and count missing days.
5. Record the temperature quality flags.
6. Fill short gaps and list the weather years usable for the replay.
7. Result.

## 1. Set up

In [1]:
import pandas as pd

from pynmdf import io_pv, io_weather
from pynmdf.config import load_config

cfg = load_config()
interim, checks = cfg["dirs"]["interim"], cfg["dirs"]["checks"]
first_year, last_year = cfg["study"]["first_year"], cfg["study"]["last_year"]

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 60)
pd.set_option("display.float_format", "{:,.1f}".format)

## 2. PV capacity at 30 June

Each workbook has a network "Total" row. The capacity column is found by its header text, because sheet names and column positions change from year to year. FY2015 and FY2016 give one capacity column; from FY2017 there are two, residential and non-residential, and the network figure is their sum.

`pv_capacity_series` stops with an error if any year differs from the expected table by more than 1 kWp, so reaching the next cell is itself the check.

In [2]:
capacity = io_pv.pv_capacity_series(cfg)
capacity.to_frame().to_parquet(interim / "pv_annual.parquet")

check = pd.DataFrame({"read_kwp": capacity, "expected_kwp": pd.Series(cfg["pv"]["expected_kwp"])})
check["difference_kwp"] = check["read_kwp"] - check["expected_kwp"]
check.to_csv(checks / "check_pv_capacity.csv")
check.T

,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
read_kwp,"229,975.3","287,107.5","345,750.2","448,682.0","619,932.1","780,243.0","1,050,389.7","1,291,295.6","1,562,557.0","1,881,669.0","2,083,689.6"
expected_kwp,"229,975.0","287,108.0","345,750.0","448,682.0","619,932.0","780,243.0","1,050,390.0","1,291,296.0","1,562,557.0","1,881,669.0","2,083,690.0"
difference_kwp,0.3,-0.5,0.2,0.0,0.1,0.0,-0.3,-0.4,-0.0,0.0,-0.4


## 3. Daily PV capacity

The daily models need a capacity for every day. Capacity is taken to follow a straight line between consecutive 30 June values. The study window starts six months before the first value and ends six months after the last, so the first and last lines are extended at their own slopes.

The table shows the two ends of the window and the values either side of them, so the extension can be checked by eye: the 30 June rows must equal section 2, in MW.

In [3]:
pv = io_pv.daily_pv(capacity, first_year, last_year)
pv.to_frame().to_parquet(interim / "pv_daily.parquet")

shown = [f"{first_year}-01-01", f"{first_year}-06-30", f"{first_year + 1}-06-30",
         f"{last_year - 1}-06-30", f"{last_year}-06-30", f"{last_year}-12-31"]
pv.loc[shown].to_frame().T

date,2015-01-01,2015-06-30,2016-06-30,2024-06-30,2025-06-30,2025-12-31
pv_mw,201.9,230.0,287.1,"1,881.7","2,083.7","2,185.5"


## 4. Weather as published, and its gaps

The three BOM files are joined by calendar date. Solar exposure is kept in MJ/m², the unit in the file header. The minimum temperature describes the early morning of its date and the maximum the afternoon of its date.

The table starts in 1990, the first year of solar exposure, because the weather replay reaches back that far. `any_variable` counts days on which at least one of the three is missing.

In [4]:
published = io_weather.read_weather(cfg)
missing = io_weather.missing_days(published)
missing.to_csv(checks / "check_weather_missing.csv")

in_window = missing.loc[first_year:last_year].sum()
print(f"Missing days, {first_year} to {last_year}:", in_window.to_dict())
print(f"Missing days, {cfg['weather']['first_year']} to {first_year - 1}:",
      missing.loc[: first_year - 1].sum().to_dict())
missing.T

Missing days, 2015 to 2025: {'tmax': 5, 'tmin': 21, 'solar': 3, 'any_variable': 27}
Missing days, 1990 to 2014: {'tmax': 1, 'tmin': 4, 'solar': 475, 'any_variable': 480}


year,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
tmax,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,2
tmin,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,4,1,3,2,2,2,6
solar,68,28,28,61,60,56,15,22,14,19,3,6,10,20,1,46,3,0,0,15,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0
any_variable,69,28,28,61,61,57,15,22,14,19,4,6,10,20,1,46,3,0,1,15,0,0,0,0,0,0,0,2,0,4,1,3,3,4,3,7


## 5. Temperature quality flags

BOM flags a temperature Y once quality control is complete and N while it is not, or when the date is uncertain. The values are used either way; the share flagged N is recorded for the limitations. Solar exposure is derived from satellite images and carries no flag.

In [5]:
quality = io_weather.temperature_quality(cfg)
quality.to_csv(checks / "check_weather_quality.csv")

flagged = quality.drop(index="missing", errors="ignore")
share_n = (flagged.loc["N"] / flagged.sum()).map("{:.1%}".format)
print("Share of published values flagged N:", share_n.to_dict())
quality

Share of published values flagged N: {'tmax': '87.7%', 'tmin': '87.6%'}


,tmax,tmin
N,3519,3499
Y,494,494
missing,5,25


## 6. Fill short gaps; weather years usable for the replay

Gaps of up to three days are filled by straight-line interpolation, and the daily mean temperature is the mean of the maximum and minimum. Longer gaps stay blank.

The weather replay in notebook 05 runs each model through whole historical weather years. A year can be used only if it has no blank day left after filling. The rolling origin trains to the year before each test year, so the list is counted at the first and the last origin.

In [6]:
weather = io_weather.fill_weather(published, cfg["weather"]["max_gap_days"])
weather.to_parquet(interim / "weather_daily.parquet")

still_blank = weather.loc[str(first_year) : str(last_year)].isna().sum()
print("Blank days left in the study window:", still_blank.to_dict())

usable = io_weather.complete_years(weather)
first_origin, last_origin = cfg["evaluation"]["first_test_year"] - 1, last_year - 1
print(f"Usable weather years up to {first_origin}: {sum(year <= first_origin for year in usable)}")
print(f"Usable weather years up to {last_origin}: {sum(year <= last_origin for year in usable)}")
print("Usable:", usable)
print("Ruled out:", sorted(set(range(cfg["weather"]["first_year"], last_year + 1)) - set(usable)))

Blank days left in the study window:

 {'tmax': 0, 'tmin': 0, 'solar': 0, 'tmean': 0}
Usable weather years up to 2018: 17
Usable weather years up to 2024: 23
Usable: [1998, 1999, 2000, 2001, 2004, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Ruled out: [1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 2002, 2003, 2005, 2009]


## 7. Result

Both driver series are built and saved. PV capacity matches the expected table in every year, and the study window has no blank weather day once gaps of up to three days are filled.

In [7]:
mw = capacity / 1000
first, last = mw.index[0], mw.index[-1]
print(f"PV capacity at 30 June: {mw[first]:,.0f} MW in {first}, {mw[last]:,.0f} MW in {last}")
print(f"Growth over the window: {mw[last] / mw[first]:.1f} times")
print(f"Daily PV series: {pv.iloc[0]:,.0f} MW on the first day, {pv.iloc[-1]:,.0f} MW on the last")
print(f"Weather: {len(weather):,} days, {weather.index[0].date()} to {weather.index[-1].date()}")
weather.loc[str(first_year) : str(last_year)].describe().loc[["min", "50%", "max"]]

PV capacity at 30 June: 230 MW in 2015, 2,084 MW in 2025
Growth over the window: 9.1 times
Daily PV series: 202 MW on the first day, 2,186 MW on the last
Weather: 13,149 days, 1990-01-01 to 2025-12-31


,tmax,tmin,solar,tmean
min,11.4,4.2,1.3,9.2
50%,23.3,15.2,14.9,19.4
max,43.7,27.5,33.1,33.2
